# Springboard Capstone Two: YouTube Trending Videos

In this project, we'll be exploring a global YouTube trending videos dataset to better understand the patterns that characterize highly visible content on the platform.

We'll be utilizing an exploratory and unsupervised approach to uncover any meaninigful structures that push videos into YouTube's "trending" feed. The notebook is structured with the typical ML lifecycle steps we've followed throughout this bootcamp. Through this initial analysis, I hope to establish a more in-depth look on YouTube's engagement metrics, and utilize these deeper patterns for additional YouTube-ML projects in the future.

*Data source*: [Kaggle - canerkonuk/youtube-trending-videos-global](https://www.kaggle.com/datasets/canerkonuk/youtube-trending-videos-global)

In [1]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler

# Since I'm working in WSL, setting MPLCONFIGDIR to a local .matplotlib directory helps with permission issues
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".matplotlib").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

# Configuring pandas for a chunkier display
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

In [2]:

# Load environment variables from .env file
env_path = Path(".env").expanduser().resolve()
load_dotenv(env_path, override=False)

data_dir = os.getenv("DATA_DIR")
data_file = os.getenv("DATA_FILE")

if not data_dir or not data_file:
    raise ValueError("DATA_DIR and DATA_FILE must both be set in .env before reading the dataset.")

csv_path = Path(data_dir).expanduser() / data_file

if not csv_path.exists():
    raise FileNotFoundError(f"CSV file not found: {csv_path}")

## 1. Data Wrangling

The `.csv` file we're working with is absolutely massive, over 18 GB worth and 800,000+ entries! So we're going to try to just load the headers first for now, and then see what fat we can trim off before feeding it into a dataframe.

In [3]:
# Read just the header row to get column names and indices
raw_columns = pd.read_csv(csv_path, nrows=0).columns.tolist()
available_columns = pd.DataFrame({"column_name": raw_columns})

print(f"Found {len(raw_columns)} columns")
available_columns

Found 28 columns


,column_name
0,video_id
1,video_published_at
2,video_trending__date
3,video_trending_country
4,channel_id
5,video_title
6,video_description
7,video_default_thumbnail
8,video_category_id
9,video_tags


Documentation on what each of these columns represent can be found at [YouTube's v3 API documentation](https://developers.google.com/youtube/v3/docs) or the data source.

The API documentation can be a bit tough to decipher, but luckily having worked with the API before I'm fairly familiar with each of these variables.

Some immediate features I would like to **avoid** are:

- `video_trending_country`: We're going to try to avoid country data since it will create too many dummy features.

- `video_title`: We'll avoid any unstructured text data for now since they will require NLP.

- `video_description`

- `video_default_thumbnail`: We'll also be avoiding image data for the same reasons as text data.

- `video_tags`: I imagine this feature could be quite rich in information it provides for us. Unfortunately, it is still text data and would only be able to be realistically processed with NLP, since tags are custom and not selected from some sort of predefined list.

- `video_dimension`: This is actually an interesting feature, since it can be a deterministic feature that seperates regular YouTube content from YouTube shorts. There has also very recently been a change in the platform to support dynamic video dimensions now. But I believe this will likely just add extra noise since the vast majority of content is the same dimension on YouTube.

- `video_definition`

- `channel_title`

- `channel_description`

- `channel_custom_url`

- `channel_country`

- `channel_have_hidden_subscribers`: A strange feature. I believe this is a boolean that indicates if channels have hidden their subscriber count. I haven't encountered any channels that actually do this, but I think this feature would just add noise, so we'll drop it.

- `channel_localized_title`

- `channel_localized_description`

In [4]:
use_cols = [
    "video_id",
    "video_published_at",
    "video_trending__date",
    "channel_id",
    "video_category_id",
    "video_duration",
    "video_licensed_content",
    "video_view_count",
    "video_like_count",
    "video_comment_count",
    "channel_published_at",
    "channel_view_count",
    "channel_subscriber_count",
    "channel_video_count",
]
date_cols = [
    "video_published_at",
    "video_trending__date",
    "channel_published_at",
]

In order to avoid potentially crashing our work environment, we'll be using a combination of the `usecols` parameter to specify which columns we want to load, and the `chunksize` parameter to load the data in smaller chunks. This allows us to work with large subsets of the data without having to load the entire dataset into memory.

In [5]:
chunk_size = 100000
chunks = pd.read_csv(
    csv_path, 
    usecols=use_cols, 
    parse_dates=date_cols, 
    chunksize=chunk_size
    )

In [6]:
chunk = next(chunks)
chunk.head()

,video_id,video_published_at,video_trending__date,channel_id,video_category_id,video_duration,video_licensed_content,video_view_count,video_like_count,video_comment_count,channel_published_at,channel_view_count,channel_subscriber_count,channel_video_count
0,bB3-CUMERIU,2024-10-11 00:00:06+00:00,2024-10-12,UCNYi_zGmR519r5gYdOKLTjQ,Music,PT2M28S,False,20535235.0,2042255.0,152933.0,2021-01-13T06:19:55.86689Z,464615150.0,11600000.0,43.0
1,5ObJt_71AYc,2024-10-11 02:59:21+00:00,2024-10-12,UCzU8-lZlRfkV3nj0RzAZdrQ,Sports,PT10M8S,True,3966042.0,NaN,2549.0,2014-02-19T20:24:31Z,399046746.0,1610000.0,4637.0
2,zfb0whgBBA8,2024-10-11 11:07:25+00:00,2024-10-12,UCgGYPnVJytkr6sVNLQ-l0zQ,Gaming,PT43M24S,True,853167.0,101155.0,10541.0,2012-08-01T16:24:26Z,114331110.0,1380000.0,314.0
3,SJfoPdeOPCQ,2024-10-11 00:10:10+00:00,2024-10-12,UCzU8-lZlRfkV3nj0RzAZdrQ,Sports,PT10M9S,True,3758707.0,NaN,3115.0,2014-02-19T20:24:31Z,399046746.0,1610000.0,4637.0
4,UVb6QOKy0bI,2024-10-09 12:30:27+00:00,2024-10-12,UCOzubmwpVZI7gD0Jf7Bk3Aw,Film & Animation,PT2M12S,True,1730189.0,67522.0,2869.0,2017-05-31T14:31:01Z,19991522.0,40600.0,56.0


In [7]:
chunk.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 14 columns):
 #   Column                    Non-Null Count   Dtype              
---  ------                    --------------   -----              
 0   video_id                  100000 non-null  str                
 1   video_published_at        100000 non-null  datetime64[us, UTC]
 2   video_trending__date      100000 non-null  datetime64[us]     
 3   channel_id                100000 non-null  str                
 4   video_category_id         99896 non-null   str                
 5   video_duration            100000 non-null  str                
 6   video_licensed_content    100000 non-null  bool               
 7   video_view_count          100000 non-null  float64            
 8   video_like_count          98438 non-null   float64            
 9   video_comment_count       99105 non-null   float64            
 10  channel_published_at      100000 non-null  str                
 11  channel_view

I can already tell that a few features will need cleaning, but I would like to pull in all of the chunks we'll be modifying first so that I can do all of the cleaning in one go. Given that the dataset contains about 800,000 entries, we'll settle for loading in 2 more chunks and work with a subset of 300,000. We'll also space out our selection since it appears that the data is sorted by `video_published_at`.

In [8]:
selected_chunks = [chunk]
target_chunks = {3, 7}

for i, chunk in enumerate(chunks):
    if i in target_chunks:
        selected_chunks.append(chunk)
    if i > max(target_chunks):
        break

df = pd.concat(selected_chunks, ignore_index=True)

In [9]:
# Now let's check out our new dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 14 columns):
 #   Column                    Non-Null Count   Dtype              
---  ------                    --------------   -----              
 0   video_id                  300000 non-null  str                
 1   video_published_at        300000 non-null  datetime64[us, UTC]
 2   video_trending__date      300000 non-null  datetime64[us]     
 3   channel_id                300000 non-null  str                
 4   video_category_id         299757 non-null  str                
 5   video_duration            300000 non-null  str                
 6   video_licensed_content    300000 non-null  bool               
 7   video_view_count          299977 non-null  float64            
 8   video_like_count          296623 non-null  float64            
 9   video_comment_count       297905 non-null  float64            
 10  channel_published_at      300000 non-null  str                
 11  channel_vie

In [10]:
# channel_published_at had a hard time converting to datetime due to mixed formats, so we'll convert it manually

df["channel_published_at"] = pd.to_datetime(
    df["channel_published_at"], 
    utc=True,
    format="ISO8601"
)

df["channel_published_at"].head()

0   2021-01-13 06:19:55.866890+00:00
1          2014-02-19 20:24:31+00:00
2          2012-08-01 16:24:26+00:00
3          2014-02-19 20:24:31+00:00
4          2017-05-31 14:31:01+00:00
Name: channel_published_at, dtype: datetime64[us, UTC]

In [11]:
df.rename(columns={"video_trending__date": "video_trending_date"}, inplace=True)
df.head()

,video_id,video_published_at,video_trending_date,channel_id,video_category_id,video_duration,video_licensed_content,video_view_count,video_like_count,video_comment_count,channel_published_at,channel_view_count,channel_subscriber_count,channel_video_count
0,bB3-CUMERIU,2024-10-11 00:00:06+00:00,2024-10-12,UCNYi_zGmR519r5gYdOKLTjQ,Music,PT2M28S,False,20535235.0,2042255.0,152933.0,2021-01-13 06:19:55.866890+00:00,464615150.0,11600000.0,43.0
1,5ObJt_71AYc,2024-10-11 02:59:21+00:00,2024-10-12,UCzU8-lZlRfkV3nj0RzAZdrQ,Sports,PT10M8S,True,3966042.0,NaN,2549.0,2014-02-19 20:24:31+00:00,399046746.0,1610000.0,4637.0
2,zfb0whgBBA8,2024-10-11 11:07:25+00:00,2024-10-12,UCgGYPnVJytkr6sVNLQ-l0zQ,Gaming,PT43M24S,True,853167.0,101155.0,10541.0,2012-08-01 16:24:26+00:00,114331110.0,1380000.0,314.0
3,SJfoPdeOPCQ,2024-10-11 00:10:10+00:00,2024-10-12,UCzU8-lZlRfkV3nj0RzAZdrQ,Sports,PT10M9S,True,3758707.0,NaN,3115.0,2014-02-19 20:24:31+00:00,399046746.0,1610000.0,4637.0
4,UVb6QOKy0bI,2024-10-09 12:30:27+00:00,2024-10-12,UCOzubmwpVZI7gD0Jf7Bk3Aw,Film & Animation,PT2M12S,True,1730189.0,67522.0,2869.0,2017-05-31 14:31:01+00:00,19991522.0,40600.0,56.0


There are still a few features left to clean:
- `video_id` and `channel_id` are not ideal as strings. I think they will likely get dropped, but I want to do a bit of exploring before we get rid of them, since I'm curious to see if there are any repeating values.
- `video_category_id` is an interesting one. I know from my experience with a previous project that there are already enumerated values we can use for these.
- `video_duration` is formatted as periodtime ISO 8601, resulting in a string. We'll have to convert that one to a numerical value, likely in total seconds.
- `video_licensed_content` can be converted to binary.

In [12]:
# Converting `video_licensed_content` to binary
df["video_licensed_content"] = df["video_licensed_content"].astype(int)

In [13]:
# Converting video_duration from ISO 8601 format to total seconds
def parse_iso8601_duration(duration_str):
    if pd.isna(duration_str):
        return np.nan
    
    duration_str = duration_str.replace("PT", "")
    hours, minutes, seconds = 0, 0, 0
    
    if "H" in duration_str:
        hours = int(duration_str.split("H")[0])
        duration_str = duration_str.split("H")[1] if "H" in duration_str else ""
    
    if "M" in duration_str:
        minutes = int(duration_str.split("M")[0])
        duration_str = duration_str.split("M")[1] if "M" in duration_str else ""
    
    if "S" in duration_str:
        seconds = int(duration_str.split("S")[0])
    
    total_seconds = hours * 3600 + minutes * 60 + seconds
    return total_seconds

In [14]:
df['video_duration_seconds'] = df['video_duration'].apply(parse_iso8601_duration)
df.drop(columns=['video_duration'], inplace=True)

The following category dictionary can be obtained via the YouTube API. Since I don't really want to mess around with APIs in this Notebook, I'm just copy/pasting it from my previous project.

In [15]:
category_df = pd.DataFrame(
    [
        {"id": 1, "title": "Film & Animation", "assignable": True},
        {"id": 2, "title": "Autos & Vehicles", "assignable": True},
        {"id": 10, "title": "Music", "assignable": True},
        {"id": 15, "title": "Pets & Animals", "assignable": True},
        {"id": 17, "title": "Sports", "assignable": True},
        {"id": 18, "title": "Short Movies", "assignable": False},
        {"id": 19, "title": "Travel & Events", "assignable": True},
        {"id": 20, "title": "Gaming", "assignable": True},
        {"id": 21, "title": "Videoblogging", "assignable": False},
        {"id": 22, "title": "People & Blogs", "assignable": True},
        {"id": 23, "title": "Comedy", "assignable": True},
        {"id": 24, "title": "Entertainment", "assignable": True},
        {"id": 25, "title": "News & Politics", "assignable": True},
        {"id": 26, "title": "Howto & Style", "assignable": True},
        {"id": 27, "title": "Education", "assignable": True},
        {"id": 28, "title": "Science & Technology", "assignable": True},
        {"id": 29, "title": "Nonprofits & Activism", "assignable": True},
        {"id": 30, "title": "Movies", "assignable": False},
        {"id": 31, "title": "Anime/Animation", "assignable": False},
        {"id": 32, "title": "Action/Adventure", "assignable": False},
        {"id": 33, "title": "Classics", "assignable": False},
        {"id": 34, "title": "Comedy", "assignable": False},
        {"id": 35, "title": "Documentary", "assignable": False},
        {"id": 36, "title": "Drama", "assignable": False},
        {"id": 37, "title": "Family", "assignable": False},
        {"id": 38, "title": "Foreign", "assignable": False},
        {"id": 39, "title": "Horror", "assignable": False},
        {"id": 40, "title": "Sci-Fi/Fantasy", "assignable": False},
        {"id": 41, "title": "Thriller", "assignable": False},
        {"id": 42, "title": "Shorts", "assignable": False},
        {"id": 43, "title": "Shows", "assignable": False},
        {"id": 44, "title": "Trailers", "assignable": False},
    ]
)

Interestingly, "Comedy" appears twice as a category. It seems there is one for when `assignable` is True and a different one for when it's false. My intution tells me that the `assignable = False` id is for licensed content. Let's have a quick look.

In [16]:
categories = df['video_category_id'].unique()
categories

<StringArray>
[               'Music',               'Sports',               'Gaming',
     'Film & Animation',       'People & Blogs',        'Entertainment',
      'News & Politics',               'Comedy',        'Howto & Style',
            'Education',      'Travel & Events',       'Pets & Animals',
 'Science & Technology',                    nan,     'Autos & Vehicles']
Length: 15, dtype: str

In [17]:
licensed_comedy_count = (
    df.loc[df['video_category_id'] == "Comedy", 'video_licensed_content']
    .value_counts(dropna=False)
    .sort_index(ascending=False)
)
licensed_comedy_count

video_licensed_content
1    16026
0      472
Name: count, dtype: int64

In [18]:
assignable_category_lookup = (
    category_df.loc[category_df['assignable'], ['title', 'id']]
    .drop_duplicates(subset='title')
    .set_index('title')['id']
)

df['category_id'] = df['video_category_id'].map(assignable_category_lookup)

comedy_mask = df['video_category_id'].eq("Comedy")
df.loc[comedy_mask, 'category_id'] = np.where(
    df.loc[comedy_mask, 'video_licensed_content'].eq(1),
    34,
    23,
)

df['category_id'] = df['category_id'].astype('Int64')
df[['video_category_id', 'category_id']].head()

,video_category_id,category_id
0,Music,10
1,Sports,17
2,Gaming,20
3,Sports,17
4,Film & Animation,1


In [19]:
df.drop(columns=['video_category_id'], inplace=True)
df.head()

,video_id,video_published_at,video_trending_date,channel_id,video_licensed_content,video_view_count,video_like_count,video_comment_count,channel_published_at,channel_view_count,channel_subscriber_count,channel_video_count,video_duration_seconds,category_id
0,bB3-CUMERIU,2024-10-11 00:00:06+00:00,2024-10-12,UCNYi_zGmR519r5gYdOKLTjQ,0,20535235.0,2042255.0,152933.0,2021-01-13 06:19:55.866890+00:00,464615150.0,11600000.0,43.0,148,10
1,5ObJt_71AYc,2024-10-11 02:59:21+00:00,2024-10-12,UCzU8-lZlRfkV3nj0RzAZdrQ,1,3966042.0,NaN,2549.0,2014-02-19 20:24:31+00:00,399046746.0,1610000.0,4637.0,608,17
2,zfb0whgBBA8,2024-10-11 11:07:25+00:00,2024-10-12,UCgGYPnVJytkr6sVNLQ-l0zQ,1,853167.0,101155.0,10541.0,2012-08-01 16:24:26+00:00,114331110.0,1380000.0,314.0,2604,20
3,SJfoPdeOPCQ,2024-10-11 00:10:10+00:00,2024-10-12,UCzU8-lZlRfkV3nj0RzAZdrQ,1,3758707.0,NaN,3115.0,2014-02-19 20:24:31+00:00,399046746.0,1610000.0,4637.0,609,17
4,UVb6QOKy0bI,2024-10-09 12:30:27+00:00,2024-10-12,UCOzubmwpVZI7gD0Jf7Bk3Aw,1,1730189.0,67522.0,2869.0,2017-05-31 14:31:01+00:00,19991522.0,40600.0,56.0,132,1
